In [31]:
#step 1

import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge

PATH = "whl_2025_ordered.csv"
df = pd.read_csv(PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values per column:")
display(df.isna().sum().sort_values(ascending=False))

print("\nTOI stats:")
display(df["toi"].describe())


Dataset shape: (25827, 26)

Columns:
['game_id', 'record_id', 'home_team', 'away_team', 'went_ot', 'home_off_line', 'home_def_pairing', 'away_off_line', 'away_def_pairing', 'home_goalie', 'away_goalie', 'toi', 'home_assists', 'home_shots', 'home_xg', 'home_max_xg', 'home_goals', 'away_assists', 'away_shots', 'away_xg', 'away_max_xg', 'away_goals', 'home_penalties_committed', 'home_penalty_minutes', 'away_penalties_committed', 'away_penalty_minutes']

First 5 rows:


,game_id,record_id,home_team,away_team,went_ot,home_off_line,home_def_pairing,away_off_line,away_def_pairing,home_goalie,...,home_goals,away_assists,away_shots,away_xg,away_max_xg,away_goals,home_penalties_committed,home_penalty_minutes,away_penalties_committed,away_penalty_minutes
0,game_1,record_1,thailand,pakistan,0,PP_kill_dwn,PP_kill_dwn,PP_up,PP_up,player_id_142,...,0,2,9,1.4645,0.2166,1,7,14,1,2
1,game_1,record_2,thailand,pakistan,0,second_off,second_def,second_off,second_def,player_id_142,...,0,2,1,0.0928,0.0928,1,0,0,0,0
2,game_1,record_3,thailand,pakistan,0,first_off,second_def,second_off,second_def,player_id_142,...,0,0,2,0.1880,0.0940,0,0,0,0,0
3,game_1,record_4,thailand,pakistan,0,second_off,first_def,second_off,first_def,player_id_142,...,0,0,1,0.0727,0.0727,0,0,0,0,0
4,game_1,record_5,thailand,pakistan,0,second_off,second_def,first_off,second_def,player_id_142,...,0,2,1,0.0769,0.0769,1,0,0,0,0



Missing values per column:


game_id                     0
record_id                   0
away_penalties_committed    0
home_penalty_minutes        0
home_penalties_committed    0
away_goals                  0
away_max_xg                 0
away_xg                     0
away_shots                  0
away_assists                0
home_goals                  0
home_max_xg                 0
home_xg                     0
home_shots                  0
home_assists                0
toi                         0
away_goalie                 0
home_goalie                 0
away_def_pairing            0
away_off_line               0
home_def_pairing            0
home_off_line               0
went_ot                     0
away_team                   0
home_team                   0
away_penalty_minutes        0
dtype: int64


TOI stats:


count    25827.000000
mean       190.194902
std        153.702230
min          0.010000
25%         92.645000
50%        156.190000
75%        237.985000
max       1559.670000
Name: toi, dtype: float64

In [32]:
#step 2

valid_lines = ["first_off", "second_off"]

# True even-strength-ish filter: BOTH teams must be in first/second line labels
df_even = df[
    df["home_off_line"].isin(valid_lines) &
    df["away_off_line"].isin(valid_lines) &
    (df["toi"] > 0)
].copy()

print("df_even shape:", df_even.shape, " (out of", df.shape, ")")
print("\nHome line labels kept:")
display(df_even["home_off_line"].value_counts())
print("\nAway line labels kept:")
display(df_even["away_off_line"].value_counts())


df_even shape: (20991, 26)  (out of (25827, 26) )

Home line labels kept:


home_off_line
first_off     10496
second_off    10495
Name: count, dtype: int64


Away line labels kept:


away_off_line
first_off     10496
second_off    10495
Name: count, dtype: int64

In [33]:
df_even = df_even.reset_index(drop=False).rename(columns={"index": "segment_id"})

n = len(df_even)
cut = int(0.8 * n)

train_seg = set(df_even.loc[:cut-1, "segment_id"])
test_seg  = set(df_even.loc[cut:, "segment_id"])

print("Segments:", n, "Train:", len(train_seg), "Test:", len(test_seg))

Segments: 20991 Train: 16792 Test: 4199


In [34]:
home_rows = pd.DataFrame({
    "segment_id": df_even["segment_id"],
    "off_team": df_even["home_team"],
    "off_line": df_even["home_off_line"],
    "def_team": df_even["away_team"],
    "def_pair": df_even["away_def_pairing"],
    "goalie": df_even["away_goalie"],
    "is_home_off": 1,
    "xg_for": df_even["home_xg"],
    "g_for": df_even["home_goals"],
    "xg_against": df_even["away_xg"],
    "g_against": df_even["away_goals"],
    "toi": df_even["toi"],
})

away_rows = pd.DataFrame({
    "segment_id": df_even["segment_id"],
    "off_team": df_even["away_team"],
    "off_line": df_even["away_off_line"],
    "def_team": df_even["home_team"],
    "def_pair": df_even["home_def_pairing"],
    "goalie": df_even["home_goalie"],
    "is_home_off": 0,
    "xg_for": df_even["away_xg"],
    "g_for": df_even["away_goals"],
    "xg_against": df_even["home_xg"],
    "g_against": df_even["home_goals"],
    "toi": df_even["toi"],
})

rapm_rows = pd.concat([home_rows, away_rows], ignore_index=True)
rapm_rows["xg_rate"] = rapm_rows["xg_for"] / rapm_rows["toi"]

In [35]:
#step 4
league_rate_per_team = rapm_rows["xg_for"].sum() / rapm_rows["toi"].sum()   # xG/sec per team
league_xg60_per_team = league_rate_per_team * 3600

print("League rate per team (xG/sec):", league_rate_per_team)
print("League xG/60 per team:", league_xg60_per_team)


League rate per team (xG/sec): 0.0006276268343921356
League xG/60 per team: 2.2594566038116883


In [36]:
#step 5

team_line_weighted = (
    rapm_rows.groupby(["off_team", "off_line"], as_index=False)
            .agg(line_xg_sum=("xg_for", "sum"),
                 line_toi_sum=("toi", "sum"))
)

team_line_weighted["line_xg_rate"] = team_line_weighted["line_xg_sum"] / team_line_weighted["line_toi_sum"]
team_line_weighted["off_line_effect"] = team_line_weighted["line_xg_rate"] - league_rate_per_team

print("Top 15 offensive (team, line) effects:")
display(team_line_weighted.sort_values("off_line_effect", ascending=False).head(15))

print("Bottom 15 offensive (team, line) effects:")
display(team_line_weighted.sort_values("off_line_effect", ascending=True).head(15))


Top 15 offensive (team, line) effects:


,off_team,off_line,line_xg_sum,line_toi_sum,line_xg_rate,off_line_effect
34,pakistan,first_off,86.6960,104327.03,0.000831,0.000203
55,thailand,second_off,86.0541,105633.67,0.000815,0.000187
35,pakistan,second_off,82.7578,103639.13,0.000799,0.000171
46,serbia,first_off,84.0457,107210.71,0.000784,0.000156
12,guatemala,first_off,85.1347,108679.36,0.000783,0.000156
54,thailand,first_off,84.5984,107994.96,0.000783,0.000156
1,brazil,second_off,81.2621,105451.28,0.000771,0.000143
58,uk,first_off,86.7948,114550.76,0.000758,0.000130
60,usa,first_off,80.7497,106785.16,0.000756,0.000129
50,south_korea,first_off,80.3274,106292.55,0.000756,0.000128


Bottom 15 offensive (team, line) effects:


,off_team,off_line,line_xg_sum,line_toi_sum,line_xg_rate,off_line_effect
57,uae,second_off,45.6282,111169.82,0.000410,-0.000217
25,mongolia,second_off,46.7614,107028.18,0.000437,-0.000191
24,mongolia,first_off,50.6237,110749.96,0.000457,-0.000171
45,saudi_arabia,second_off,51.1025,111049.08,0.000460,-0.000167
20,kazakhstan,first_off,52.6645,113395.46,0.000464,-0.000163
21,kazakhstan,second_off,51.8104,111525.28,0.000465,-0.000163
52,switzerland,first_off,52.1687,109754.28,0.000475,-0.000152
18,indonesia,first_off,56.7233,112896.07,0.000502,-0.000125
40,philippines,first_off,60.8357,118618.82,0.000513,-0.000115
41,philippines,second_off,59.2622,115236.91,0.000514,-0.000113


In [37]:
#step 6

team_def_weighted = (
    rapm_rows.groupby(["def_team", "def_pair"], as_index=False)
            .agg(allowed_xg_sum=("xg_for", "sum"),   # opponent offense xG
                 def_toi_sum=("toi", "sum"))
)

team_def_weighted["xg_allowed_rate"] = team_def_weighted["allowed_xg_sum"] / team_def_weighted["def_toi_sum"]
team_def_weighted["def_pair_effect"] = league_rate_per_team - team_def_weighted["xg_allowed_rate"]  # + = better

print("Top 15 defensive pairings:")
display(team_def_weighted.sort_values("def_pair_effect", ascending=False).head(15))

print("Bottom 15 defensive pairings:")
display(team_def_weighted.sort_values("def_pair_effect", ascending=True).head(15))


Top 15 defensive pairings:


,def_team,def_pair,allowed_xg_sum,def_toi_sum,xg_allowed_rate,def_pair_effect
0,brazil,first_def,46.7833,107004.94,0.000437,0.000190
23,mexico,second_def,54.5467,107967.95,0.000505,0.000122
12,guatemala,first_def,55.7155,110249.41,0.000505,0.000122
42,rwanda,first_def,54.9108,104871.47,0.000524,0.000104
38,peru,first_def,59.2360,112758.99,0.000525,0.000102
29,netherlands,second_def,58.6912,111515.57,0.000526,0.000101
28,netherlands,first_def,59.4127,112076.26,0.000530,0.000098
4,china,first_def,61.2017,114823.35,0.000533,0.000095
52,switzerland,first_def,58.2415,109240.28,0.000533,0.000094
55,thailand,second_def,56.7666,106359.85,0.000534,0.000094


Bottom 15 defensive pairings:


,def_team,def_pair,allowed_xg_sum,def_toi_sum,xg_allowed_rate,def_pair_effect
49,singapore,second_def,83.1413,107480.08,0.000774,-0.000146
60,usa,first_def,79.1840,106347.99,0.000745,-0.000117
25,mongolia,second_def,79.4032,106902.18,0.000743,-0.000115
59,uk,second_def,82.1636,111001.63,0.000740,-0.000113
11,germany,second_def,79.6012,109106.55,0.000730,-0.000102
7,ethiopia,second_def,77.4256,106779.48,0.000725,-0.000097
47,serbia,second_def,74.8964,103982.72,0.000720,-0.000093
19,indonesia,second_def,79.8788,110903.22,0.000720,-0.000093
62,vietnam,first_def,77.6434,109955.72,0.000706,-0.000079
33,oman,second_def,73.1345,103612.16,0.000706,-0.000078


In [38]:
#step 7

goalie_weighted = (
    rapm_rows.groupby("goalie", as_index=False)
            .agg(
                xg_faced=("xg_for", "sum"),
                goals_allowed=("g_for", "sum"),
                toi_sum=("toi", "sum")
            )
)

goalie_weighted["xg_rate"] = goalie_weighted["xg_faced"] / goalie_weighted["toi_sum"]
goalie_weighted["ga_rate"] = goalie_weighted["goals_allowed"] / goalie_weighted["toi_sum"]
goalie_weighted["goalie_effect"] = goalie_weighted["xg_rate"] - goalie_weighted["ga_rate"]  # + = saves above expected

print("Top 10 goalies (save more than expected):")
display(goalie_weighted.sort_values("goalie_effect", ascending=False).head(10))

print("Bottom 10 goalies (allow more than expected):")
display(goalie_weighted.sort_values("goalie_effect", ascending=True).head(10))


Top 10 goalies (save more than expected):


,goalie,xg_faced,goals_allowed,toi_sum,xg_rate,ga_rate,goalie_effect
18,player_id_257,140.2054,100,221202.48,0.000634,0.000452,0.000182
24,player_id_38,152.0975,112,233855.73,0.000650,0.000479,0.000171
9,player_id_16,135.1439,111,212480.77,0.000636,0.000522,0.000114
22,player_id_293,144.9328,123,219761.28,0.000660,0.000560,0.000100
25,player_id_40,153.7209,132,225694.69,0.000681,0.000585,0.000096
7,player_id_148,118.1039,102,223591.83,0.000528,0.000456,0.000072
4,player_id_14,110.2451,95,211962.50,0.000520,0.000448,0.000072
3,player_id_123,161.3095,146,218510.63,0.000738,0.000668,0.000070
31,player_id_93,152.4296,137,222724.86,0.000684,0.000615,0.000069
14,player_id_218,124.2388,109,226229.14,0.000549,0.000482,0.000067


Bottom 10 goalies (allow more than expected):


,goalie,xg_faced,goals_allowed,toi_sum,xg_rate,ga_rate,goalie_effect
29,player_id_80,123.2893,156,209214.16,0.000589,0.000746,-0.000156
11,player_id_208,146.6494,177,219800.50,0.000667,0.000805,-0.000138
16,player_id_232,136.9541,161,218817.29,0.000626,0.000736,-0.000110
0,player_id_103,115.1878,138,216742.69,0.000531,0.000637,-0.000105
17,player_id_25,139.5335,162,223487.55,0.000624,0.000725,-0.000101
20,player_id_259,138.9364,155,223873.26,0.000621,0.000692,-0.000072
10,player_id_192,144.3565,157,224920.74,0.000642,0.000698,-0.000056
8,player_id_152,151.2039,163,210817.46,0.000717,0.000773,-0.000056
6,player_id_142,116.3722,127,213628.63,0.000545,0.000594,-0.000050
28,player_id_75,153.4500,164,219946.33,0.000698,0.000746,-0.000048


In [39]:
#step 8:

# Team offense (TOI-weighted)
team_off = team_line_weighted.copy()
team_off["weighted_off"] = team_off["off_line_effect"] * team_off["line_toi_sum"]
team_off = team_off.groupby("off_team", as_index=False).agg(
    weighted_off=("weighted_off", "sum"),
    toi_sum=("line_toi_sum", "sum")
)
team_off["team_off_effect"] = team_off["weighted_off"] / team_off["toi_sum"]
team_off = team_off.rename(columns={"off_team": "team"})[["team", "team_off_effect"]]

# Team defense (TOI-weighted)
team_def = team_def_weighted.copy()
team_def["weighted_def"] = team_def["def_pair_effect"] * team_def["def_toi_sum"]
team_def = team_def.groupby("def_team", as_index=False).agg(
    weighted_def=("weighted_def", "sum"),
    toi_sum=("def_toi_sum", "sum")
)
team_def["team_def_effect"] = team_def["weighted_def"] / team_def["toi_sum"]
team_def = team_def.rename(columns={"def_team": "team"})[["team", "team_def_effect"]]

# Team goalie usage: goalies belong to the defending team (def_team) in rapm_rows
team_goalie_use = rapm_rows.groupby(["def_team", "goalie"], as_index=False).agg(goalie_toi=("toi", "sum"))
team_goalie_use = team_goalie_use.rename(columns={"def_team": "team"}).merge(
    goalie_weighted[["goalie", "goalie_effect"]], on="goalie", how="left"
)
team_goalie_use["goalie_effect"] = team_goalie_use["goalie_effect"].fillna(0.0)
team_goalie_use["weighted_goalie"] = team_goalie_use["goalie_effect"] * team_goalie_use["goalie_toi"]

team_goalie = team_goalie_use.groupby("team", as_index=False).agg(
    weighted_goalie=("weighted_goalie", "sum"),
    toi_sum=("goalie_toi", "sum")
)
team_goalie["team_goalie_effect"] = team_goalie["weighted_goalie"] / team_goalie["toi_sum"]
team_goalie = team_goalie[["team", "team_goalie_effect"]]

team_simple = team_off.merge(team_def, on="team").merge(team_goalie, on="team", how="left")
team_simple["team_net_rating"] = (
    team_simple["team_off_effect"] + team_simple["team_def_effect"] + team_simple["team_goalie_effect"]
)
team_simple = team_simple.sort_values("team_net_rating", ascending=False).reset_index(drop=True)
team_simple.insert(0, "rank", team_simple.index + 1)

print("TEAM POWER RANKINGS (components, even-strength-ish):")
display(team_simple)


TEAM POWER RANKINGS (components, even-strength-ish):


,rank,team,team_off_effect,team_def_effect,team_goalie_effect,team_net_rating
0,1,brazil,1.343188e-04,0.000108,0.000072,0.000314
1,2,pakistan,1.871875e-04,0.000053,0.000012,0.000252
2,3,thailand,1.712010e-04,0.000083,-0.000050,0.000204
3,4,india,-4.724734e-07,-0.000006,0.000182,0.000175
4,5,china,5.289173e-05,0.000052,0.000044,0.000149
5,6,peru,-1.981256e-05,0.000078,0.000067,0.000126
6,7,uk,7.587399e-05,-0.000053,0.000096,0.000119
7,8,iceland,4.486368e-06,-0.000008,0.000114,0.000110
8,9,netherlands,-6.286246e-05,0.000099,0.000072,0.000109
9,10,panama,2.067345e-05,-0.000032,0.000100,0.000089


In [40]:
rapm_rows["off_unit"] = rapm_rows["off_team"].astype(str) + "__" + rapm_rows["off_line"].astype(str)
rapm_rows["def_unit"] = rapm_rows["def_team"].astype(str) + "__" + rapm_rows["def_pair"].astype(str)

train = rapm_rows[rapm_rows["segment_id"].isin(train_seg)].copy()
test  = rapm_rows[rapm_rows["segment_id"].isin(test_seg)].copy()

X_train = pd.get_dummies(train[["off_unit","def_unit","goalie","is_home_off"]], drop_first=True)
y_train = train["xg_rate"].values
w_train = train["toi"].values

X_test = pd.get_dummies(test[["off_unit","def_unit","goalie","is_home_off"]], drop_first=True)
y_test = test["xg_rate"].values
w_test = test["toi"].values

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

model = Ridge(alpha=50.0, fit_intercept=True, random_state=0)
model.fit(X_train, y_train, sample_weight=w_train)

yhat = model.predict(X_test)

def wmae(y, yhat, w):
    return np.sum(w * np.abs(y - yhat)) / np.sum(w)

def wrmse(y, yhat, w):
    return np.sqrt(np.sum(w * (y - yhat)**2) / np.sum(w))

print("TEST wMAE xG/60:", wmae(y_test, yhat, w_test) * 3600)
print("TEST wRMSE xG/60:", wrmse(y_test, yhat, w_test) * 3600)

coef = pd.Series(model.coef_, index=X_train.columns)
intercept = float(model.intercept_)
print("Intercept baseline xG_rate:", intercept)
print("Home-off coefficient:", float(coef.get("is_home_off", np.nan)))

# Attach predictions to test rows
test = test.copy()
test["pred_xg_rate"] = yhat
test["pred_xg"] = test["pred_xg_rate"] * test["toi"]

# Aggregate to game level (2 rows per segment_id)
game_pred = (
    test.groupby("segment_id", as_index=False)
        .agg(
            pred_xg_home=("pred_xg", lambda x: x.iloc[0]),
            pred_xg_away=("pred_xg", lambda x: x.iloc[1]),
            actual_goals_home=("g_for", lambda x: x.iloc[0]),
            actual_goals_away=("g_for", lambda x: x.iloc[1]),
        )
)

game_pred["pred_diff"] = game_pred["pred_xg_home"] - game_pred["pred_xg_away"]
game_pred["actual_diff"] = game_pred["actual_goals_home"] - game_pred["actual_goals_away"]

game_pred["pred_win"] = (game_pred["pred_diff"] > 0).astype(int)
game_pred["actual_win"] = (game_pred["actual_diff"] > 0).astype(int)

accuracy = (game_pred["pred_win"] == game_pred["actual_win"]).mean()

print("Win prediction accuracy:", round(accuracy * 100, 2), "%")

TEST wMAE xG/60: 1.7199412797701752
TEST wRMSE xG/60: 2.2708804376717073
Intercept baseline xG_rate: 0.0007048468950662564
Home-off coefficient: 5.932837460709985e-05
Win prediction accuracy: 40.7 %


In [41]:
#step 10:

def get_coef(name: str) -> float:
    return float(coef[name]) if name in coef.index else 0.0

teams = sorted(pd.unique(pd.concat([df_even["home_team"], df_even["away_team"]], ignore_index=True)))

off_usage = rapm_rows.groupby(["off_team", "off_line"], as_index=False).agg(toi=("toi", "sum"))
def_usage = rapm_rows.groupby(["def_team", "def_pair"], as_index=False).agg(toi=("toi", "sum"))

rows = []
for t in teams:
    off_t = off_usage[off_usage["off_team"] == t].copy()
    off_t["unit_name"] = off_t["off_team"].astype(str) + "__" + off_t["off_line"].astype(str)
    off_t["unit_coef"] = off_t["unit_name"].map(lambda u: get_coef(f"off_unit_{u}"))
    team_off = (off_t["unit_coef"] * off_t["toi"]).sum() / off_t["toi"].sum()

    def_t = def_usage[def_usage["def_team"] == t].copy()
    def_t["unit_name"] = def_t["def_team"].astype(str) + "__" + def_t["def_pair"].astype(str)
    def_t["unit_coef_raw"] = def_t["unit_name"].map(lambda u: get_coef(f"def_unit_{u}"))
    team_def = - (def_t["unit_coef_raw"] * def_t["toi"]).sum() / def_t["toi"].sum()

    rows.append({"team": t, "off_rapm": team_off, "def_rapm": team_def, "net_rapm": team_off + team_def})

team_rank_xg = pd.DataFrame(rows)
team_rank_xg["off_rapm_per60"] = team_rank_xg["off_rapm"] * 3600
team_rank_xg["def_rapm_per60"] = team_rank_xg["def_rapm"] * 3600
team_rank_xg["net_rapm_per60"] = team_rank_xg["net_rapm"] * 3600

team_rank_xg = team_rank_xg.sort_values("net_rapm_per60", ascending=False).reset_index(drop=True)
team_rank_xg.insert(0, "rank", team_rank_xg.index + 1)

print("TEAM POWER RANKINGS (xG RAPM, per 60):")
display(team_rank_xg[["rank", "team", "off_rapm_per60", "def_rapm_per60", "net_rapm_per60"]])


TEAM POWER RANKINGS (xG RAPM, per 60):


,rank,team,off_rapm_per60,def_rapm_per60,net_rapm_per60
0,1,pakistan,0.295341,0.054399,0.349740
1,2,thailand,0.126928,0.082404,0.209332
2,3,mexico,-0.131399,0.312379,0.180981
3,4,south_korea,-0.099638,0.021853,-0.077785
4,5,serbia,0.031379,-0.137871,-0.106492
5,6,guatemala,-0.263455,0.048220,-0.215235
6,7,uk,-0.138522,-0.079073,-0.217596
7,8,china,-0.294520,0.031335,-0.263184
8,9,oman,-0.244542,-0.050048,-0.294590
9,10,brazil,0.023645,-0.325886,-0.302241


In [42]:
#step 11:

# Team goalie GSAx per60 (TOI-weighted), goalies belong to def_team
team_goalie_use2 = rapm_rows.groupby(["def_team", "goalie"], as_index=False).agg(toi=("toi", "sum"))
team_goalie_use2 = team_goalie_use2.rename(columns={"def_team": "team"}).merge(
    goalie_weighted[["goalie", "goalie_effect"]], on="goalie", how="left"
)
team_goalie_use2["goalie_effect"] = team_goalie_use2["goalie_effect"].fillna(0.0)
team_goalie_use2["weighted"] = team_goalie_use2["goalie_effect"] * team_goalie_use2["toi"]

team_goalie2 = team_goalie_use2.groupby("team", as_index=False).agg(weighted=("weighted", "sum"), toi=("toi", "sum"))
team_goalie2["team_goalie_gsax_per60"] = (team_goalie2["weighted"] / team_goalie2["toi"]) * 3600

LAMBDA = 0.35

final_rank = team_rank_xg.merge(team_goalie2[["team", "team_goalie_gsax_per60"]], on="team", how="left")
final_rank["team_goalie_gsax_per60"] = final_rank["team_goalie_gsax_per60"].fillna(0.0)

final_rank["final_power_per60"] = final_rank["net_rapm_per60"] + LAMBDA * final_rank["team_goalie_gsax_per60"]
final_rank = final_rank.sort_values("final_power_per60", ascending=False).reset_index(drop=True)
final_rank["rank"] = final_rank.index + 1

display(final_rank[[
    "rank","team","off_rapm_per60","def_rapm_per60","net_rapm_per60","team_goalie_gsax_per60","final_power_per60"
]])

final_rank.to_csv("power_rankings_whl_2025.csv", index=False)

,rank,team,off_rapm_per60,def_rapm_per60,net_rapm_per60,team_goalie_gsax_per60,final_power_per60
0,1,pakistan,0.295341,0.054399,0.349740,0.043453,0.364948
1,2,thailand,0.126928,0.082404,0.209332,-0.179096,0.146648
2,3,mexico,-0.131399,0.312379,0.180981,-0.378901,0.048365
3,4,uk,-0.138522,-0.079073,-0.217596,0.346465,-0.096333
4,5,serbia,0.031379,-0.137871,-0.106492,0.012816,-0.102006
5,6,china,-0.294520,0.031335,-0.263184,0.159496,-0.207361
6,7,brazil,0.023645,-0.325886,-0.302241,0.258925,-0.211617
7,8,panama,-0.328464,-0.024786,-0.353250,0.359290,-0.227499
8,9,guatemala,-0.263455,0.048220,-0.215235,-0.070723,-0.239988
9,10,india,-0.437712,-0.031583,-0.469294,0.654330,-0.240279


In [43]:
#step 12:

tot = rapm_rows.groupby("off_team", as_index=False).agg(
    xg_for=("xg_for", "sum"),
    g_for=("g_for", "sum"),
    xg_against=("xg_against", "sum"),
    g_against=("g_against", "sum"),
    toi=("toi", "sum"),
).rename(columns={"off_team": "team"})

tot["finishing_for"] = tot["g_for"] - tot["xg_for"]                 # + = scoring more than xG
tot["finishing_against"] = tot["xg_against"] - tot["g_against"]     # + = allowing fewer goals than xG
tot["finishing_net"] = tot["finishing_for"] + tot["finishing_against"]

print("Best finishing (likely hot/lucky):")
display(tot.sort_values("finishing_net", ascending=False).head(10))

print("Worst finishing (likely cold/unlucky):")
display(tot.sort_values("finishing_net", ascending=True).head(10))


Best finishing (likely hot/lucky):


,team,xg_for,g_for,xg_against,g_against,toi,finishing_for,finishing_against,finishing_net
8,india,138.7281,140,140.2054,100,221202.48,1.2719,40.2054,41.4773
7,iceland,134.3119,148,135.1439,111,212480.77,13.6881,24.1439,37.8320
3,ethiopia,139.8203,165,147.5275,138,214878.40,25.1797,9.5275,34.7072
20,philippines,120.0979,110,152.0975,112,233855.73,-10.0979,40.0975,29.9996
9,indonesia,114.1018,124,152.4296,137,222724.86,9.8982,15.4296,25.3278
0,brazil,161.5039,169,110.2451,95,211962.50,7.4961,15.2451,22.7412
19,peru,137.5053,143,124.2388,109,226229.14,5.4947,15.2388,20.7335
18,panama,142.4713,135,144.9328,123,219761.28,-7.4713,21.9328,14.4615
14,netherlands,126.2767,123,118.1039,102,223591.83,-3.2767,16.1039,12.8272
12,mongolia,97.3851,101,156.7276,151,217778.14,3.6149,5.7276,9.3425


Worst finishing (likely cold/unlucky):


,team,xg_for,g_for,xg_against,g_against,toi,finishing_for,finishing_against,finishing_net
4,france,136.2989,125,146.6494,177,219800.50,-11.2989,-30.3506,-41.6495
1,canada,143.5865,128,136.9541,161,218817.29,-15.5865,-24.0459,-39.6324
11,mexico,150.2319,134,115.1878,138,216742.69,-16.2319,-22.8122,-39.0441
13,morocco,144.1871,128,139.5335,162,223487.55,-16.1871,-22.4665,-38.6536
28,uae,108.1131,105,138.9364,155,223873.26,-3.1131,-16.0636,-19.1767
6,guatemala,146.9474,135,125.7543,130,216117.99,-11.9474,-4.2457,-16.1931
10,kazakhstan,104.4749,102,144.3565,157,224920.74,-2.4749,-12.6435,-15.1184
21,rwanda,125.4968,113,124.7863,127,210299.47,-12.4968,-2.2137,-14.7105
25,south_korea,150.5437,172,123.2893,156,209214.16,21.4563,-32.7107,-11.2544
31,vietnam,125.9904,114,145.4891,144,218257.16,-11.9904,1.4891,-10.5013
